In [ ]:
import pandas as pd
import selfies as sf
import exmol
import time

from rdkit import Chem
from rdkit.Chem import Descriptors
from tdc.single_pred import Tox

from dglgcn import largest_mol

# packages for plotting
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf', 'IBMPlexMono-Regular.ttf')
fe = font_manager.FontEntry(
    fname='IBMPlexMono-Regular.ttf',
    name='plexmono')
font_manager.fontManager.ttflist.append(fe)
plt.rcParams.update({'axes.facecolor':'#f5f4e9',
            'grid.color' : '#AAAAAA',
            'axes.edgecolor':'#333333',
            'figure.facecolor':'#FFFFFF',
            'axes.grid': False,
            'axes.prop_cycle':   plt.cycler('color', plt.cm.Dark2.colors),
            'font.family': fe.name,
            'figure.figsize': (3.5,3.5 / 1.2),
            'ytick.left': True,
            'xtick.bottom': True   ,
            'figure.dpi': 300
           })

In [ ]:
data = Tox(name='LD50_Zhu')
df = data.get_data()
df_clean = df.copy()
df_clean['Drug'] = df_clean['Drug'].apply(largest_mol)
print(df_clean.shape)
df_clean.head()

In [ ]:
def count_selfies_tokens(smiles):
    try:
        sel = sf.encoder(smiles)
        return len(list(sf.split_selfies(sel)))
    except:
        return None


print(f"Dataset size: {len(df_clean)}")
print(f"Label range: {df_clean.Y.min():.2f} to {df_clean.Y.max():.2f}")

df_clean['selfies_tokens'] = df_clean.Drug.apply(count_selfies_tokens)

# molecular weight and heavy atom count
df_clean['mol'] = df_clean.Drug.apply(Chem.MolFromSmiles)
df_clean['MW'] = df_clean.mol.apply(Descriptors.MolWt)
df_clean['heavy_atoms'] = df_clean.mol.apply(lambda m: m.GetNumHeavyAtoms())

# fragments check
df_clean['has_fragments'] = df_clean.Drug.apply(lambda s: '.' in s)
print(f"\nSMILES with fragments: {df_clean.has_fragments.sum()}")

print(df_clean['selfies_tokens'].describe())
print(f"Tokens < 5: {(df_clean.selfies_tokens < 5).sum()}")
print(f"Tokens < 10: {(df_clean.selfies_tokens < 10).sum()}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), dpi=150)

# LD50 label distribution
axs[0].hist(df_clean.Y, bins=50, edgecolor='none')
axs[0].set_xlabel('LD50 (log mg/kg)')
axs[0].set_ylabel('Count')
axs[0].set_title('LD50 Label Distribution')

# SMILES length distribution
axs[1].hist(df_clean.selfies_tokens, bins=50, edgecolor='none')
axs[1].set_xlabel('Selfies Token Count')
axs[1].set_ylabel('Count')
axs[1].set_title('Selfies Token Distribution')

plt.tight_layout()
plt.savefig('ld50_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

### STONED Analysis on LD50 Dataset

In [ ]:
# reproduce figure 6 in wellawatte et al. 2022
initial_smiles = 'S=C=Nc1ccc(Br)cc1' # from ld50 data, df.Drug[4]
print('SMILES sequence: ',initial_smiles)

exps = []
spaces = []
for i in [1, 3, 5]:
    stoned_kwargs = {
        "num_samples": 2500,
        "alphabet": exmol.get_basic_alphabet(),
        "min_mutations": i,
        "max_mutations": i,
    }
    space = exmol.sample_space(
        initial_smiles, f=lambda x: 0, batched=False, preset='medium', stoned_kwargs=stoned_kwargs, quiet=True,
    )

    spaces.append(space)
    e = exmol.rcf_explain(space, nmols=2)
    if len(exps) == 0:
        exps.append(e[0])
    for ei in e:
        if not ei.is_origin and "Decrease" in ei.label:
            ei.label = f"Mutations = {i}"
            exps.append(ei)
            break

# plotting
fig, axs = plt.subplots(1, 3, figsize=(8, 3), dpi=180, squeeze=True, sharey=True)
for i, n in enumerate([1, 3, 5]):
    axs[i].hist([e.similarity for e in spaces[i][1:]], bins=99, edgecolor="none")
    axs[i].set_title(f"Mutations = {n}")
    axs[i].set_xlim(0, 1)
plt.tight_layout()
#plt.savefig("mutation-hist.png", bbox_inches="tight", dpi=180)

In [ ]:
q25 = df_clean.selfies_tokens.quantile(0.25)
q75 = df_clean.selfies_tokens.quantile(0.75)

df_clean['size_group'] = pd.cut(
    df_clean.selfies_tokens,
    bins=[0, q25, q75, float('inf')],
    labels=['Small', 'Medium', 'Large']
)

print(f"Q25: {q25}, Q75: {q75}")
print(df_clean.size_group.value_counts())

# sample molecules from each group
n_sample = 30  # per group
samples = {}
for group in ['Small', 'Medium', 'Large']:
    group_df = df_clean[df_clean.size_group == group]
    samples[group] = group_df.sample(n=min(n_sample, len(group_df)), random_state=42).Drug.tolist()

# run STONED analysis per group per mutation level
results = {group: {n: [] for n in [1, 3, 5]} for group in ['Small', 'Medium', 'Large']}

for group, smiles_list in samples.items():
    for smi in smiles_list:
        for n_mut in [1, 3, 5]:
            stoned_kwargs = {
                "num_samples": 100,
                "alphabet": exmol.get_basic_alphabet(),
                "min_mutations": n_mut,
                "max_mutations": n_mut,
            }
            space = exmol.sample_space(
                smi, f=lambda x: 0, batched=False,
                preset='medium', stoned_kwargs=stoned_kwargs, quiet=True
            )
            similarities = [e.similarity for e in space[1:]]
            results[group][n_mut].extend(similarities)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(10, 8), dpi=150, sharey=True, sharex=True)

for i, group in enumerate(['Small', 'Medium', 'Large']):
    for j, n_mut in enumerate([1, 3, 5]):
        axs[i][j].hist(results[group][n_mut], bins=50, edgecolor='none')
        axs[i][j].set_xlim(0, 1)
        if i == 0:
            axs[i][j].set_title(f'Mutations = {n_mut}')
        if j == 0:
            axs[i][j].set_ylabel(f'{group}\n Mol. Count')
        if i == 2:
            axs[i][j].set_xlabel('Tanimoto Similarity')

plt.suptitle('Tanimoto Similarity Distribution by Molecule Size and Mutation Level (LD50 Dataset)')
plt.tight_layout()
plt.savefig('stoned_analysis_ld50.pdf', bbox_inches='tight')
plt.show()

### STONED analysis on Lipophilicity dataset

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv",
    "./lipophilicity.csv",
)
df_lipo = pd.read_csv("./lipophilicity.csv")
# lipodata['SMILES_length'] = lipodata['smiles'].apply(lambda x: len(x))
lipodata = list(zip(df_lipo.smiles,df_lipo.exp))

In [ ]:
df_lipo['selfies_tokens'] = df_lipo.smiles.apply(count_selfies_tokens)
print(df_lipo['selfies_tokens'].describe())
print(f"Tokens < 5: {(df_lipo.selfies_tokens < 5).sum()}")
print(f"Tokens < 10: {(df_lipo.selfies_tokens < 10).sum()}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), dpi=150)

# LD50 label distribution
axs[0].hist(df_lipo.exp, bins=50, edgecolor='none')
axs[0].set_xlabel('logD')
axs[0].set_ylabel('Count')
axs[0].set_title('Lipophilicity Label Distribution')

# SMILES length distribution
axs[1].hist(df_lipo.selfies_tokens, bins=50, edgecolor='none')
axs[1].set_xlabel('Selfies Token Count')
axs[1].set_ylabel('Count')
axs[1].set_title('Selfies Token Distribution')

plt.tight_layout()
plt.show()

In [ ]:
q25 = df_lipo.selfies_tokens.quantile(0.25)
q75 = df_lipo.selfies_tokens.quantile(0.75)

df_lipo['size_group'] = pd.cut(
    df_lipo.selfies_tokens,
    bins=[0, q25, q75, float('inf')],
    labels=['Small', 'Medium', 'Large']
)

print(f"Q25: {q25}, Q75: {q75}")
print(df_lipo.size_group.value_counts())

# sample molecules from each group
n_sample = 30  # per group
samples = {}
for group in ['Small', 'Medium', 'Large']:
    group_df = df_lipo[df_lipo.size_group == group]
    samples[group] = group_df.sample(n=min(n_sample, len(group_df)), random_state=42).smiles.tolist()

# run STONED analysis per group per mutation level
results = {group: {n: [] for n in [1, 3, 5]} for group in ['Small', 'Medium', 'Large']}

for group, smiles_list in samples.items():
    for smi in smiles_list:
        for n_mut in [1, 3, 5]:
            stoned_kwargs = {
                "num_samples": 100,
                "alphabet": exmol.get_basic_alphabet(),
                "min_mutations": n_mut,
                "max_mutations": n_mut,
            }
            space = exmol.sample_space(
                smi, f=lambda x: 0, batched=False,
                preset='medium', stoned_kwargs=stoned_kwargs, quiet=True
            )
            similarities = [e.similarity for e in space[1:]]
            results[group][n_mut].extend(similarities)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(10, 8), dpi=150, sharey=True, sharex=True)

for i, group in enumerate(['Small', 'Medium', 'Large']):
    for j, n_mut in enumerate([1, 3, 5]):
        axs[i][j].hist(results[group][n_mut], bins=50, edgecolor='none')
        axs[i][j].set_xlim(0, 1)
        if i == 0:
            axs[i][j].set_title(f'Mutations = {n_mut}')
        if j == 0:
            axs[i][j].set_ylabel(f'{group}\n Mol. Count')
        if i == 2:
            axs[i][j].set_xlabel('Tanimoto Similarity')

plt.suptitle('Tanimoto Similarity Distribution by Molecule Size and Mutation Level (LD50 Dataset)')
plt.tight_layout()
plt.savefig('stoned_analysis_lipo.pdf', bbox_inches='tight')
plt.show()

### Generate molecule replacements, filtering with Tanimoto similarities

In [ ]:
def get_noised_SMILES(smi, min_score, max_score, num_samples, preset='medium', ignore_fail_error=True, max_tries=10):
    # get a different SMILES with similarity score between min_score and max_score
    # first come first serve
    if min_score == 1.0 and max_score == 1.0:
        return smi, 1.0, True
    method_kwargs = {
        "num_samples": num_samples,
        "min_mutations": 1,
        "max_mutations": 1,
    }
    num_tries = 0
    while num_tries < max_tries:
        examples = exmol.sample_space(
            smi,
            preset=preset,
            f=lambda x: 0,
            batched=False,
            quiet=True,
            method_kwargs=method_kwargs
        )
        smiles = [e.smiles for e in examples[1:]] # ignore first with 1.0 score
        scores = [e.similarity for e in examples[1:]]
        for smile, score in zip(smiles, scores):
            if min_score < score < max_score:
                return smile, score, True,num_tries+1
        if max_score <= 0.3:
            # increase mutations for higher chance with low similarities
            method_kwargs['max_mutations'] += 1
        method_kwargs['num_samples'] += 10
        num_tries += 1

    msg = f'Cannot find another SMILES between {min_score} and {max_score} for this SMILES!'
    msg += f'\nSMILES: {smi}'
    if ignore_fail_error == False:
        raise Exception(msg)
    return smi, 1.0, False, num_tries

def noise_all_dataset(data, scorerange, num_samples=15, preset='medium'):
    minscore, maxscore = scorerange
    noised_smiles_count = 0
    fail_count = 0
    top_tries = 0
    for smi, label in data:
        smi = largest_mol(smi)
        if minscore == 1.0 and maxscore == 1.0:
            score = 1
            num_tries = 0
        else:
            noised_smiles, score, success, num_tries = get_noised_SMILES(
                smi, minscore, maxscore, num_samples=num_samples, preset=preset
            )
            if num_tries > top_tries:
                top_tries = num_tries
            if success == False:
                fail_count += 1
            smi = noised_smiles
            noised_smiles_count += 1
    return fail_count, top_tries

def noise_some_dataset(data, scorerange, threshold, targetregion, preset='medium'):
    minscore, maxscore = scorerange
    noised_smiles_count = 0
    fail_count = 0
    for smi, label in data:
        smi = largest_mol(smi)
        if minscore == 1.0 and maxscore == 1.0:
            score = 1
        elif targetregion == 'below':
            if label < threshold:
                noised_smiles, score, success, _ = get_noised_SMILES(
                    smi, minscore, maxscore, num_samples=15, preset=preset
                )
                if success == False:
                    fail_count += 1
                smi = noised_smiles
                noised_smiles_count += 1

        elif targetregion == 'above':
            if label > threshold:
                noised_smiles, score, success, _ = get_noised_SMILES(
                    smi, minscore, maxscore, num_samples=15, preset=preset
                )
                if success == False:
                    fail_count += 1
                smi = noised_smiles
                noised_smiles_count += 1
        else:
            raise Exception("'targetregion' can only take 'below' or 'above'")

    print(f'Noised {noised_smiles_count-fail_count} SMILES out of {len(data)} total')
    print(f'Out of {noised_smiles_count} noisy SMILES, {fail_count}'
              ' failures to generate SMILES within desired similarity scores. '
              f'Original SMILES are used instead.')
    return noised_smiles_count, fail_count

In [ ]:
import pandas as pd
import time

def benchmark_noising_performance(data, score_ranges, num_samples=15, preset='medium'):
    results_list = []

    for min_score, max_score in score_ranges:
        print(f'Noising SMILES between {min_score} and {max_score}...')
        start_time = time.time()

        fail_count, num_tries = noise_all_dataset(
            data=data,
            scorerange=(min_score, max_score),
            num_samples=num_samples,
            preset=preset,
        )

        end_time = time.time()
        elapsed_time = end_time - start_time

        results_list.append({
            'Score Range': f'{min_score}-{max_score}',
            '# Failures': fail_count,
            'Time (s)': elapsed_time,
            '# Tries': num_tries
        })
        results = pd.DataFrame(results_list)

    return results


In [ ]:
score_intervals = [
    [1.0, 1.0], # no-noise control
    [0.8, 1.0],
    [0.7, 0.8],
    [0.6, 0.7],
    [0.5, 0.6],
    [0.45, 0.5],
    [0.4, 0.45],
    [0.35, 0.4],
    [0.3, 0.35],
    [0.25, 0.3],
    [0.2, 0.25],
    [0.15, 0.2],
    [0.1, 0.15],
    [0.05, 0.1],
    [0, 0.05],
]

In [ ]:
ld50_data = list(zip(df_clean.Drug, df_clean.Y))

In [ ]:
benchmark_noising_performance(ld50_data[:], score_intervals, num_samples=15)

In [ ]:
benchmark_noising_performance(lipodata, score_intervals, num_samples=15)

### Lipophilicity distribution: originals vs STONED replacements
Histogram of experimental logD (originals) overlaid with Crippen logP of STONED-generated replacements at representative similarity ranges. Dashed lines mark censor thresholds for each split.

In [ ]:
from rdkit.Chem import Crippen
from dglgcn import compute_threshold_from_split

# censor thresholds for each split
labels_list = df_lipo['exp'].tolist()
thresholds = {
    split: compute_threshold_from_split(labels_list, split, 'above')
    for split in [0.1, 0.5, 0.9]
}
print('Censor thresholds:', thresholds)

# sample molecules — full run is slow, 150 gives a reasonable distribution
n_sample = 150
sample_df = df_lipo.sample(n=n_sample, random_state=42)

SIM_RANGES = [
    ([0.8, 1.0],  'Replacement [0.8-1.0] (noise=0.1)'),
    ([0.3, 0.4],  'Replacement [0.3-0.4] (noise=0.65)'),
    ([0.0, 0.05], 'Replacement [0.0-0.05] (noise=0.975)'),
]

for (min_s, max_s), label in SIM_RANGES:
    midpoint = (min_s + max_s) / 2
    print(f'[{min_s}-{max_s}]  midpoint={midpoint:.3f}  noise level={round(1 - midpoint, 3)}')

replacements = {label: [] for _, label in SIM_RANGES}

for _, row in sample_df.iterrows():
    smi = largest_mol(row['smiles'])
    for (min_s, max_s), label in SIM_RANGES:
        rep_smi, score, success, _ = get_noised_SMILES(
            smi, min_s, max_s, num_samples=15, preset='medium'
        )
        if success:
            mol = Chem.MolFromSmiles(rep_smi)
            if mol is not None:
                replacements[label].append(Crippen.MolLogP(mol))

for label, vals in replacements.items():
    print(f'{label}: {len(vals)} replacements')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(df_lipo['exp'], bins=50, alpha=0.5, density=True,
        label='Original (exp logD)', color='C0')

colors = ['C1', 'C2', 'C3']
for ((min_s, max_s), label), color in zip(SIM_RANGES, colors):
    if replacements[label]:
        ax.hist(replacements[label], bins=50, alpha=0.4, density=True,
                label=label, color=color)

# censor threshold lines
linestyles = ['--', '-.', ':']
for (split, thresh), ls in zip(thresholds.items(), linestyles):
    ax.axvline(thresh, linestyle=ls, color='black', linewidth=1.2,
               label=f'{int(split * 100)}% split (logD={thresh:.2f})')

ax.set_xlabel('logD / Crippen logP')
ax.set_ylabel('Density')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.savefig('stoned_lipo_histogram.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# same plot but Crippen logP for originals too
original_logp = []
for smi in df_lipo['smiles']:
    mol = Chem.MolFromSmiles(largest_mol(smi))
    if mol is not None:
        original_logp.append(Crippen.MolLogP(mol))

fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(original_logp, bins=50, alpha=0.5, density=True,
        label='Original (Crippen logP)', color='C0')

for ((min_s, max_s), label), color in zip(SIM_RANGES, colors):
    if replacements[label]:
        ax.hist(replacements[label], bins=50, alpha=0.4, density=True,
                label=label, color=color)

ax.set_xlabel('Crippen logP')
ax.set_ylabel('Density')
ax.set_xlim(-3,9)
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.savefig('stoned_lipo_histogram_crippen.pdf', bbox_inches='tight')
plt.show()